In [ ]:
from pathlib import Path
import re
from urllib.parse import unquote
from scapy.all import IP, Raw, TCP, sniff

# Базовые паттерны XSS (без учета регистра)
XSS_PATTERNS = re.compile(
    r"(<script.*?>|on\w+\s*=|javascript:|alert\s*\(|document\.cookie)",
    re.IGNORECASE,
)


def detect_xss(packet):
    # Проверяем наличие IP, TCP и полезной нагрузки (Raw)
    if packet.haslayer(IP) and packet.haslayer(TCP) and packet.haslayer(Raw):
        # Фильтруем HTTP-запросы (порт 80 или альтернативный 8080)
        if packet[TCP].dport in (80, 8080):
            raw_data = packet[Raw].load.decode(errors="ignore")
            # Декодируем URL-encoded символы (например, %3C -> <)
            decoded_data = unquote(raw_data)

            if XSS_PATTERNS.search(decoded_data):
                print(
                    f"[XSS Alert] Source: {packet[IP].src} -> Dest:"
                    f" {packet[IP].dst}"
                )
                print(decoded_data[:300])  # Выводим только начало пакета
                print("-" * 40)


if __name__ == "__main__":
    # Укажите имя вашего файла (если он лежит в той же папке, что и скрипт)
    # Или полный путь, например: Path(r"C:\Users\User\Desktop\dump.pcap")
    pcap_file = Path(r"C:\Users\Sergey\Desktop\постпление меня\Учеба 1 семестр\Питон\Тема WWW\captured_http.pcap")

    if not pcap_file.exists():
        print(f"Ошибка: файл не найден по пути {pcap_file.resolve()}")
    else:
        print(f"Анализ файла {pcap_file.name} на наличие XSS...")
        # offline= читает пакеты из файла, store=0 не забивает оперативную память
        sniff(offline=str(pcap_file), prn=detect_xss, store=0)
        print("Анализ завершен.")